# 阶段3：基于内容的电影推荐系统构建

基于processed_movies.csv数据集，使用TF-IDF和余弦相似度构建电影推荐系统。

In [ ]:
# 导入所需库
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# 加载数据
df = pd.read_csv(r"D:\电影分析项目\处理后数据\processed_movies.csv")
print(f"数据加载完成: {df.shape[0]}行, {df.shape[1]}列")

## 1. 文本特征向量化（TF-IDF）

In [ ]:
# 使用TfidfVectorizer对combined_features进行向量化
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'].fillna(''))
print(f"TF-IDF特征矩阵形状: {tfidf_matrix.shape}")

## 2. 余弦相似度计算

In [ ]:
# 计算所有电影之间的余弦相似度
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"相似度矩阵形状: {cosine_sim.shape}")

# 创建电影标题到索引的映射
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
print(f"唯一电影标题数量: {len(indices)}")

## 3. 推荐函数实现

In [ ]:
def recommend_movies(title, cosine_sim=cosine_sim, df=df, indices=indices, top_n=10):
    """
    基于内容过滤的电影推荐函数
    参数:
        title: 电影标题
        top_n: 推荐数量（默认10部）
    返回:
        推荐结果DataFrame，包含电影标题、评分、类型、导演、票房、相似度
    """
    if title not in indices:
        return f"错误：电影 '{title}' 未在数据集中找到。"

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    similarities = [i[1] for i in sim_scores]

    results = df.iloc[movie_indices][['title', 'vote_average', 'genres', 'director', 'revenue', 'vote_count']].copy()
    results['similarity'] = [round(s, 4) for s in similarities]
    results['revenue'] = results['revenue'].apply(lambda x: f"${x:,.0f}")
    results = results.reset_index(drop=True)
    results.index = results.index + 1

    return results

## 4. 推荐效果验证

In [ ]:
# 测试电影列表
test_movies = ['Avatar', 'The Dark Knight', 'Titanic', 'Forrest Gump', 'The Matrix']

for movie in test_movies:
    print(f"\n{'='*60}")
    print(f"  推荐：{movie}")
    print('='*60)
    result = recommend_movies(movie)
    if isinstance(result, pd.DataFrame):
        display(result)
    else:
        print(result)

## 5. 结果合理性分析

### Avatar的推荐结果分析
- 推荐了Alien系列（同类型科幻/动作）、Spaceballs（科幻喜剧）等
- 这些电影都包含科幻、动作元素，与Avatar的核心特征匹配

### The Dark Knight的推荐结果分析
- 推荐了蝙蝠侠系列电影（Batman Begins、The Dark Knight Rises等）
- 推荐结果高度合理，共享相同的角色世界观和犯罪/动作类型

### Titanic的推荐结果分析
- 推荐了海难题材（Poseidon、In the Heart of the Sea）和爱情片（The Notebook）
- 基于深海/灾难/爱情等共同特征进行推荐

### Forrest Gump的推荐结果分析
- 推荐了越战题材电影（The Deer Hunter、Platoon、Born on the Fourth of July）
- 基于战争/历史/剧情等共同特征

### The Matrix的推荐结果分析
- 推荐了Matrix续集和Wachowski姐妹的其他作品
- 同导演、同世界观电影的相似度最高，推荐效果最佳

## 6. 保存模型文件

In [ ]:
# 保存相似度矩阵和电影索引映射
model_dir = r"D:\电影分析项目\模型文件"

with open(f'{model_dir}/similarity_matrix.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)
print("已保存: similarity_matrix.pkl")

with open(f'{model_dir}/movie_indices.pkl', 'wb') as f:
    pickle.dump(indices, f)
print("已保存: movie_indices.pkl")